# ML-08 — Train and Compare the Model (Content Refresh lane)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yowxy/Flyrank-Internship-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Lane: **Content Refresh — "which page should an editor fix FIRST?"** (framed in `w02_ml_task_framing.ipynb`, baseline in `w04_baseline_score.ipynb`). Worked in order. Simple words, honest numbers.

> Working with an AI assistant? It read `skills/README.md` and loaded `training-honest-models` + `flyrank/flyrank-data` for this task.

## 1. Method choice and why

**Question shape: yes/no with an observed label** (`is_declining_label = (trend_direction == "down")`), used as a **ranked queue** — the classifier's probability becomes the priority score. Per the skill's method table, that shape starts with **Logistic Regression, then Random Forest**: readable → stronger. I run exactly that ladder, with a depth-limited **Decision Tree** in the middle as the readable checkpoint (a tree I can print teaches more than an opaque model a few points stronger). No Gradient Boosting: on 30k rows with 32 wildly uneven clients, a third ensemble adds tuning surface without answering the lane question, and the skill says to add complexity only when the comparison earns it.

**Why these three earn their place:** (a) LogReg (balanced, scaled) is the linear reference — if it already beats the rule, the signal is additive and I say so. (b) Tree (max_depth=5, min_samples_leaf=50) shows which 2–3 splits carry the queue, printable for an editor. (c) Random Forest (300 trees, max_depth=10, min_samples_leaf=25, balanced_subsample) tests whether combining the tangled signals — demand, rank, engagement, freshness, content type — beats hand thresholds on unseen clients. All evaluated at **precision@K** (the queue metric) alongside ROC-AUC (the ranking metric), same split, same metric as the baseline.

In [1]:
# §1 — CODE: setup, shared label definition, base rate, library versions. Local + Colab.
import os, json
import numpy as np
import pandas as pd
import sklearn

def _repo_root():
    d = os.getcwd()
    while d != "/" and not os.path.isdir(os.path.join(d, "work", "notebooks")):
        d = os.path.dirname(d)
    return d
ROOT = _repo_root()
RAW = os.path.join(ROOT, "data", "raw", "content_refresh_anonymized.csv")
OUT_JSON = os.path.join(ROOT, "work", "outputs", "w05_model_receipts.json")
SEED = 42

df = pd.read_csv(RAW)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)  # eval only, never a feature
BASE = float(df["is_declining_label"].mean())
print(f"rows: {len(df):,} | clients: {df['client_id'].nunique()} | base rate: {BASE:.4f}")
print(f"libraries: sklearn {sklearn.__version__} | pandas {pd.__version__} | numpy {np.__version__} | seed {SEED}")

rows: 30,000 | clients: 32 | base rate: 0.5421
libraries: sklearn 1.9.0 | pandas 3.0.5 | numpy 2.5.2 | seed 42


## 2. Split design

**Grouped by client (client-holdout), seed 42.** 32 clients → 20% held out = 6 never-shown clients (2,325 rows test / 27,675 train). Why this split is honest for this question: (a) per-client history depth and decline mix differ wildly (one test client declines at 0.22, another at 0.62; train base 0.555 vs test base 0.391 — measured below, not assumed), so a row-shuffle would leak each client's house style into both sides and flatter every model; (b) deployment means scoring pages for clients the model has never seen, so the test must be clients, not rows; (c) the 90-day aggregates necessarily *contain* the label's 30-day input windows (the track's stated snapshot convention) — on held-out clients that overlap cannot memorise an answer, which is exactly why Week-4's in-sample baseline must be recomputed here rather than quoted.

**Leakage quarantine (asserted in code):** never features = `trend_pct`, `trend_direction`, `is_declining_label` (label machinery), the six `*_last_30d / *_prev_30d` window columns, or the IDs (`content_id`, `client_id` — grouping only). `provider_used` / `model_used` are excluded as provenance, not content signals. Gotcha handling: `avg_position == 0` means no data → kept as value plus a `pos_missing` flag; `scroll_rate` / `ai_traffic_pct` values above 100 are kept (different measurement systems, not a bug); systematic missingness gets `has_keyword` / `has_wordcount` flags instead of a blind fillna(0).

In [2]:
# §2 — CODE: grouped split + quarantine asserts. Verdicts above must match these numbers.
rng = np.random.default_rng(SEED)
clients = df["client_id"].astype(str).unique()
perm = rng.permutation(clients)
NTEST = max(1, int(round(len(perm) * 0.2)))
TEST_CLIENTS = set(perm[:NTEST])
test_mask = df["client_id"].isin(TEST_CLIENTS).to_numpy()
tr_idx = np.where(~test_mask)[0]; te_idx = np.where(test_mask)[0]
y = df["is_declining_label"].astype(int)
assert y.iloc[tr_idx].nunique() == 2 and y.iloc[te_idx].nunique() == 2, "split left one side single-class"
print(f"train: {len(tr_idx):,} rows | test: {len(te_idx):,} rows ({NTEST} held-out clients)")
print(f"train base: {y.iloc[tr_idx].mean():.4f} | test base: {y.iloc[te_idx].mean():.4f} (shift is real — report it, don't hide it)")
print("test-client mix (n, decline rate):")
print(df.iloc[te_idx].groupby("client_id").agg(n=("is_declining_label", "size"), rate=("is_declining_label", "mean")).round(3).to_string())

BANNED = ["trend_pct", "trend_direction", "is_declining_label", "impressions_last_30d",
          "clicks_last_30d", "sessions_last_30d", "impressions_prev_30d",
          "clicks_prev_30d", "sessions_prev_30d", "content_id", "client_id",
          "provider_used", "model_used"]
NUM = ["search_volume", "competition", "cpc", "word_count", "char_count",
       "days_with_impressions", "days_with_sessions", "content_age_days",
       "days_since_last_update", "ctr", "avg_position", "engagement_rate",
       "scroll_rate", "ai_traffic_pct", "log_impressions_90d", "log_clicks_90d",
       "log_sessions_90d", "log_ai_sessions_90d", "has_keyword", "has_wordcount", "pos_missing"]
CAT = ["competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
       "word_count_tier", "char_count_tier", "impression_tier", "position_tier"]
assert not set(NUM + CAT) & set(BANNED), "leakage: banned column in feature list"
print(f"OK: {len(NUM)} numeric + {len(CAT)} categorical feature groups; none banned; IDs grouping-only")

train: 27,675 rows | test: 2,325 rows (6 held-out clients)
train base: 0.5548 | test base: 0.3910 (shift is real — report it, don't hide it)
test-client mix (n, decline rate):
                      n   rate
client_id                     
client_0b918943df    35  0.486
client_1a6562590e     3  0.000
client_4fc82b26ae    32  0.531
client_98a3ab7c34   118  0.619
client_d4735e3a26  1106  0.220
client_f74efabef1  1031  0.542
OK: 21 numeric + 9 categorical feature groups; none banned; IDs grouping-only


## 3. Train + compare vs my baseline

**Same data, same split, same metrics — the non-negotiable table.** The Week-4 rule (`stale90 × visible500 × weak_snippet(ctr<0.5, 1≤pos≤20) × (21−pos)`, in-sample P@50=0.90) is **recomputed on this exact client-holdout split**, not quoted from its in-sample receipt — quoting it would compare a memorised score against honest ones. Metrics: ROC-AUC (queue ordering across thresholds) and precision@20/50/200 (what an editor's sprint actually contains), plus the test base rate so every number has a denominator.

**Result (measured, seed 42): the rule collapses off its home clients while every learned model holds.** Baseline test AUC = 0.501 (random) with P@50 = 0.420 — its thresholds were tuned to the clients it was measured on. Random Forest wins the ladder at AUC 0.747 / P@200 0.705 (test base 0.391); the depth-5 tree already reaches AUC 0.742, and even linear LogReg reaches 0.696 — above the rule at every K. Against my pre-registered bar (AUC ≥ 0.70 AND P@200 ≥ 0.65 on held-out clients): RF **passes both**; the tree passes AUC but misses P@200 (0.595); LogReg misses both. Complexity verdict: the tree earns readability, RF earns the queue — Gradient Boosting is not needed to answer this lane.

In [3]:
# §3 — CODE: honest features, baseline recomputed on SAME split, LR -> tree -> RF, comparison table.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

for c in ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d"]:
    df[c] = np.log1p(df[c.replace("log_", "")].fillna(0))  # heavy-tailed traffic
df["has_keyword"] = df["search_volume"].notna().astype(int)
df["has_wordcount"] = df["word_count"].notna().astype(int)
df["pos_missing"] = (df["avg_position"] == 0).astype(int)  # 0 = no data, flagged not dropped

X = pd.concat([df[NUM].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0),
               pd.get_dummies(df[CAT].fillna("unknown").astype(str), dtype=float)], axis=1)
print(f"feature matrix: {X.shape[0]:,} x {X.shape[1]} (seed {SEED})")

def precision_at_k(yt, scores, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(yt)[order[:k]].mean())

# Week-4 rule, verbatim logic, scored on THIS split (never quoted in-sample)
stale = (df["days_since_last_update"] >= 90).astype(int)
vis = (df["impressions_90d"] >= 500).astype(int)
weak = ((df["ctr"] < 0.5) & (df["avg_position"] >= 1) & (df["avg_position"] <= 20)).astype(int)
bscore = ((stale * vis * weak).astype(int) * (21 - df["avg_position"].clip(lower=0)))
yte = y.iloc[te_idx].reset_index(drop=True)
bte = bscore.iloc[te_idx].reset_index(drop=True)

models = {
    "logreg": Pipeline([("sc", StandardScaler()),
                         ("m", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED))]),
    "tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=SEED),
    "rf": RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                                 n_estimators=300, n_jobs=-1, random_state=SEED),
}
probs = {}
for name, m in models.items():
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    probs[name] = m.predict_proba(X.iloc[te_idx])[:, 1]

rows = [("baseline_rule",
         roc_auc_score(yte, bte),
         precision_at_k(yte, bte, 20),
         precision_at_k(yte, bte, 50),
         precision_at_k(yte, bte, 200))]
for name in ["logreg", "tree", "rf"]:
    p = probs[name]
    rows.append((name, roc_auc_score(y.iloc[te_idx], p),
                 precision_at_k(y.iloc[te_idx], p, 20),
                 precision_at_k(y.iloc[te_idx], p, 50),
                 precision_at_k(y.iloc[te_idx], p, 200)))
comp = pd.DataFrame(rows, columns=["model", "AUC", "P@20", "P@50", "P@200"]).round(3)
comp["test_base"] = round(float(y.iloc[te_idx].mean()), 3)
print(comp.to_string(index=False))
print(f"(reference: baseline IN-SAMPLE was P@50=0.90 — quoted only to show the drop, never compared)")

with open(OUT_JSON, "w") as f:
    json.dump({"seed": SEED, "train_rows": int(len(tr_idx)), "test_rows": int(len(te_idx)),
               "n_test_clients": int(NTEST), "train_base": round(float(y.iloc[tr_idx].mean()), 4),
               "test_base": round(float(y.iloc[te_idx].mean()), 4),
               "table": comp.to_dict(orient="records")}, f, indent=2, sort_keys=True)
print(f"wrote: {OUT_JSON}")

feature matrix: 30,000 x 60 (seed 42)


        model   AUC  P@20  P@50  P@200  test_base
baseline_rule 0.501   0.4  0.42  0.395      0.391
       logreg 0.696   0.4  0.52  0.545      0.391
         tree 0.742   0.5  0.56  0.595      0.391
           rf 0.747   0.7  0.78  0.705      0.391
(reference: baseline IN-SAMPLE was P@50=0.90 — quoted only to show the drop, never compared)
wrote: /home/iklil/Internship/FLyrank/Assingnment1/flyrank-ml-internship-starter/work/outputs/w05_model_receipts.json


## 4. Errors and interpretation

**What it leans on (permutation importance on held-out clients, RF):** `days_with_impressions` (0.029) and `log_impressions_90d` (0.016) top the shuffle test, then `log_clicks_90d`, `ctr`, `days_with_sessions`, and the `impression_tier_low` flag — all demand/visibility quantities, which plausibly relate to decline (a page losing impression-days is visibly decaying). Built-in impurity agrees on the same family (`days_with_impressions`, `log_impressions_90d`, `avg_position`, `content_age_days`). Sanity check against leakage: no top-15 feature is label machinery — nothing suspiciously perfect (top shuffle-delta is 0.029, not 0.30), so this reads as signal, not a leak. Notably the model leans on *sustained visibility*, while the baseline leaned on *staleness × snippet* — complementary stories, and visibility travels better across clients.

**Where it is most wrong:** errors concentrate on `keyword article` pages (40.8% wrong, n=1,367) vs `feedly article` (23.0%, n=958) — the model has no keyword-competition signal for feedly rows (100% missing by construction) yet does *better* there, so the misses are about keyword pages' noisier demand, not the missingness flags. By rank, `top_3` pages are nearly solved (9.3% wrong) while `page_3_5`/`striking`/`deep` run 37–43% wrong: the queue is trustworthy at the top, noisy in the tail. At the 0.5 line, 44.7% of flagged pages are false alarms (editor hours wasted) against 20.9% missed declines among passed pages — precision, not recall, remains the cost to watch.

**Three concrete wrong cases and why they're hard:** (a) FP — 3,026 impressions, ZERO clicks, position 35.9, p=0.75, label healthy: looks exactly like a dying page (invisible snippet, deep rank), but the label says stable — either demand shifted without crossing −20% or the page was already flat; no 90-day aggregate separates "dead snippet" from "stable at zero". (b) FP — 761 impressions, position 25.1, CTR 0.39, p=0.74, label healthy: the model reads deep-rank + thin clicks as decay, but the page held its window — the −20% label line cuts through near-identical pages. (c) FN — 1 impression, no position data, 1 session, p=0.05, label declining: a near-invisible page with `pos_missing` that the model files as "no signal", yet it crossed the −20% line from almost nothing — at n=1 no edit's effect is measurable anyway (the same boundary weakness the Week-4 review found in rank 7).

In [4]:
# §4 — CODE: permutation importance (RF, test clients) + error slices + 3 wrong cases.
from sklearn.inspection import permutation_importance
rf = models["rf"]
pi = permutation_importance(rf, X.iloc[te_idx], y.iloc[te_idx], n_repeats=5, random_state=SEED, n_jobs=-1)
perm_imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
print("permutation importance, top 12 (held-out clients):")
print(perm_imp.head(12).round(4).to_string())
print("\nbuilt-in impurity, top 8 (same model):")
print(pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(8).round(4).to_string())
assert not set(perm_imp.head(15).index) & set(BANNED), "banned column surfaced in importance"

t = df.iloc[te_idx].copy()
t["p"] = probs["rf"]; t["pred"] = (t["p"] >= 0.5).astype(int); t["wrong"] = (t["pred"] != t["is_declining_label"]).astype(int)
print("\nerror rate by content_type:"); print(t.groupby("content_type")["wrong"].agg(["mean", "size"]).round(3).to_string())
print("\nerror rate by position_tier:"); print(t.groupby("position_tier")["wrong"].agg(["mean", "size"]).round(3).to_string())
fp = float(((t["pred"] == 1) & (t["is_declining_label"] == 0)).sum() / max((t["pred"] == 1).sum(), 1))
fn = float(((t["pred"] == 0) & (t["is_declining_label"] == 1)).sum() / max((t["pred"] == 0).sum(), 1))
print(f"\nfalse-alarm share of flagged: {fp:.3f} | missed-decline share of passed: {fn:.3f}")
show = ["impressions_90d", "clicks_90d", "sessions_90d", "avg_position", "ctr",
        "days_since_last_update", "content_type", "main_intent", "p", "is_declining_label"]
print("\n3 wrong cases (2 false alarms, 1 missed decline):")
cases = pd.concat([t[(t["wrong"] == 1) & (t["is_declining_label"] == 0)].nlargest(2, "p"),
                   t[(t["wrong"] == 1) & (t["is_declining_label"] == 1)].nsmallest(1, "p")])
print(cases[show].round(2).to_string(index=False))

permutation importance, top 12 (held-out clients):
days_with_impressions       0.0292
log_impressions_90d         0.0157
log_clicks_90d              0.0065
ctr                         0.0052
days_with_sessions          0.0052
impression_tier_low         0.0049
scroll_rate                 0.0032
engagement_rate             0.0026
content_age_days            0.0024
impression_tier_moderate    0.0016
log_sessions_90d            0.0010
age_tier_181-365            0.0010

built-in impurity, top 8 (same model):
days_with_impressions    0.1402
log_impressions_90d      0.1328
avg_position             0.0935
content_age_days         0.0811
char_count               0.0386
log_clicks_90d           0.0354
age_tier_365+            0.0336
ctr                      0.0328

error rate by content_type:
                  mean  size
content_type                
feedly article   0.230   958
keyword article  0.408  1367

error rate by position_tier:
                mean  size
position_tier             
deep

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (pseudonym IDs used for grouping only)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Claim discipline used:** observed (measured 90-day windows on 30k rows), directional (a refresh queue to assist editors — RF P@200=0.705 vs 0.391 base on 6 held-out clients), decision-support (single client-holdout split, seed 42; false-alarm share 0.447 disclosed, not buried). Lane: Content Refresh, locked.

In [5]:
# Self-check receipts: contract shape enforced in code.
r = json.load(open(OUT_JSON))
assert r["train_rows"] == 27675 and r["test_rows"] == 2325 and r["n_test_clients"] == 6, r
tab = {row["model"]: row for row in r["table"]}
assert tab["rf"]["AUC"] >= 0.70 and tab["rf"]["P@200"] >= 0.65, tab  # pre-registered bar
assert tab["baseline_rule"]["AUC"] < 0.60, tab  # rule must not survive the client split
assert all(set(row) == {"model", "AUC", "P@20", "P@50", "P@200", "test_base"} for row in r["table"])
assert "trend_pct" not in str(X.columns.tolist()) and "trend_direction" not in str(X.columns.tolist())
print(f"OK: RF AUC={tab['rf']['AUC']} P@200={tab['rf']['P@200']} beats rule AUC={tab['baseline_rule']['AUC']} P@200={tab['baseline_rule']['P@200']} on the same {r['test_rows']:,}-row holdout")

OK: RF AUC=0.747 P@200=0.705 beats rule AUC=0.501 P@200=0.395 on the same 2,325-row holdout
